In [ ]:
import plotnine as gg
import pandas as pd
import glob
import json
from essential.utils import PLOTNINE_DEFAULT_THEME

In [ ]:
PATH_TO_RESULTS = "/workspace/results/ecoli_rich_medium/*/*/distance_metrics_mmd.json"
all_results = []
for result_path in glob.glob(PATH_TO_RESULTS):
    with open(result_path, "r") as f:
        result = json.load(f)
    all_results.append(result)

all_results_df = pd.DataFrame(all_results).set_index("tag")

In [ ]:
SELECTED_COLUMNS = [
    "target_dist_median_ratio_of_top_50_pred_pairs_to_global",
    "target_dist_median_ratio_of_top_100_pred_pairs_to_global",
    "target_dist_median_ratio_of_top_500_pred_pairs_to_global",
    "target_dist_median_ratio_of_top_1000_pred_pairs_to_global",
]

SELECTED_TAGS = [
    "fba_moma_default_mmd",
    # "gene_graph_beta_1_mmd",
    # "gene_graph_beta_10_mmd",
    # "gene_graph_beta_50_mmd",
    # "gene_graph_beta_100_mmd",
    "gene_graph_beta_auto_mmd",
]
results_df_selected = all_results_df.loc[SELECTED_TAGS]
results_df_selected[SELECTED_COLUMNS]

In [ ]:
(
    gg.ggplot(
        results_df_selected.reset_index(),
        gg.aes(x="tag", y="target_dist_median_ratio_of_top_100_pred_pairs_to_global"),
    )
    + gg.geom_col()
    + gg.theme_minimal()
    + gg.coord_flip()
)

In [ ]:
path_to_distances = "/workspace/results/ecoli_rich_medium/gene_graph/beta_auto/distances.pkl"
metabolic_df = pd.read_pickle(path_to_distances)

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, metric="precomputed", init="random")
metabolic_df_tsne = tsne.fit_transform(metabolic_df)
metabolic_low_dim = pd.DataFrame(
    metabolic_df_tsne, index=metabolic_df.index, columns=["TSNE1", "TSNE2"]
).assign(target=metabolic_df.index)

In [ ]:
(gg.ggplot(metabolic_low_dim, gg.aes(x="TSNE1", y="TSNE2")) + gg.geom_point() + gg.theme_minimal())

In [ ]:
genes = ["eno", "glxK", "garK"]

(
    gg.ggplot(metabolic_low_dim, gg.aes(x="TSNE1", y="TSNE2"))
    + gg.geom_point()
    + gg.geom_point(metabolic_low_dim.query("target in @genes"), gg.aes(color="target"))
    + gg.theme_minimal()
)